# Подключение к Spark

Тетрадка для Jupyter на платформе (`/home/jovyan`). Локальный Windows-проект озеро не видит.

Конспект рабочей тетрадки коллеги: там Spark уже поднимается. Ниже только то, что нужно для входа. Загрузка в Oracle и бизнес-логика той тетрадки сюда не перенесены.

## Как у коллеги устроено подключение

Два входа, один keytab.

1. Секреты читаются из `/home/jovyan/.rail_connectors/Personal.env` через `load_dotenv`. Логин и пароль — переменные `USERNAME` и `PASSWORD`. В коде их нет.
2. Keytab: `/home/jovyan/.rail_connectors/user_keytab.keytab`. Это не `test_requests/tech.keytab`.
3. Impala открывается через `rail_connectors.connection.connect(to='IMPALA')`. В `kerberos` передан только `keytab_path`. Флаги `use_credentials` и `update_keytab` у коллеги закомментированы. База: `sandbox_ai`. После `imp._init_connection()` в логе было `Successfully login!` и успешный `SELECT 1`.
4. Spark **не** создаётся через `.master("yarn")` и **не** получает очередь в коде. Перед `SparkSession` каталог конфигурации платформы копируется в окружение:

```python
if "SPARK_K8S_CONF_DIR" in os.environ:
    os.environ["SPARK_CONF_DIR"] = os.environ["SPARK_K8S_CONF_DIR"]
```

Мастер, очередь и параметры Kerberos приходят из этого каталога. В билдере только два явных параметра: `spark.submit.deployMode = client` и `spark.sql.catalogImplementation = hive`. Второе нужно, чтобы `spark.table("sandbox_ai....")` и `spark.sql` видели те же таблицы, что Impala.
5. Старая сессия в том же ядре гасится через `spark.stop()` в `try/except`. Если переменной ещё нет, ловится `NameError`.
6. Чтение озера после этого: `spark.table("схема.таблица")` или `spark.sql(...)`. Запись в другую базу через JDBC в этой тетрадке не нужна.

## Почему прошлый ручной запуск падал

`SparkSession.builder.master("yarn")` без билета Kerberos доходит до HDFS и падает с `Failed to find any Kerberos tgt`. JVM умирает, Python показывает `JAVA_GATEWAY_EXITED`. В тетрадке коллеги билет появляется на шаге Impala (keytab), а мастер Spark берётся из `SPARK_CONF_DIR`, а не прописывается вручную.

## Порядок ячеек

1. Импорты, `.env`, keytab.
2. Impala и `SELECT 1`.
3. Сессия Spark.
4. Проверка: версия, `select 1`, список баз.

Пароль в тетрадку не вставлять. Если `USERNAME` или `PASSWORD` пустые, ячейка Impala остановится и напишет, чего не хватает.

## 1. Импорты и секреты

In [ ]:
import logging
import os
import subprocess

from dotenv import load_dotenv
from pyspark.sql import SparkSession
from rail_connectors.connection import connect

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("spark_connect")

load_dotenv("/home/jovyan/.rail_connectors/Personal.env")
keytab_path = "/home/jovyan/.rail_connectors/user_keytab.keytab"
username = os.getenv("USERNAME")
password = os.getenv("PASSWORD")

print("username_set", bool(username))
print("password_set", bool(password))
print("keytab_exists", os.path.isfile(keytab_path))
print("SPARK_K8S_CONF_DIR", os.environ.get("SPARK_K8S_CONF_DIR"))
print("SPARK_CONF_DIR", os.environ.get("SPARK_CONF_DIR"))

## 2. Impala

Тот же вызов, что в рабочей тетрадке. Keytab здесь нужен и для входа в Impala, и чтобы дальше Spark не пошёл в HDFS без билета.

In [ ]:
if not username or not password:
    raise RuntimeError(
        "В Personal.env нет USERNAME или PASSWORD. "
        "Файл: /home/jovyan/.rail_connectors/Personal.env"
    )
if not os.path.isfile(keytab_path):
    raise RuntimeError(f"Нет keytab: {keytab_path}")

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    kerberos={"keytab_path": keytab_path},
    user_params={"user_name": username, "password": password},
)
imp._init_connection()

with imp:
    imp.execute("SELECT 1")

logger.info("Impala: SELECT 1 прошёл")
print(subprocess.getoutput("klist"))

## 3. Сессия Spark

Если `SPARK_K8S_CONF_DIR` пустой, сессия пойдёт с системным `SPARK_CONF_DIR`. У коллеги рабочий путь — подменить его каталогом платформы. Мастер и очередь в этой ячейке не задаются.

In [ ]:
logger.info("Инициализация Spark сессии")

if "SPARK_K8S_CONF_DIR" in os.environ:
    os.environ["SPARK_CONF_DIR"] = os.environ["SPARK_K8S_CONF_DIR"]
    logger.info("SPARK_CONF_DIR = %s", os.environ["SPARK_CONF_DIR"])
else:
    logger.warning("SPARK_K8S_CONF_DIR не найден. SPARK_CONF_DIR не переопределён.")

try:
    spark.stop()
    logger.info("Текущая Spark сессия остановлена")
except NameError:
    logger.info("Spark сессия ещё не создана")
except Exception as e:
    logger.info("Нет живой сессии Spark: %s", e)

spark = (
    SparkSession.builder
    .appName("acq-dashboard-mvp")
    .config("spark.submit.deployMode", "client")
    .config("spark.sql.catalogImplementation", "hive")
    .getOrCreate()
)

logger.info("Spark сессия инициализирована")
print("version", spark.version)
print("ui", spark.sparkContext.uiWebUrl)

## 4. Проверка, что сессия считает

`select 1` не трогает таблицы. `show databases` только проверяет, что каталог Hive открылся. Полную витрину отсюда не запускать.

In [ ]:
ok = spark.sql("select 1 as ok")
ok.persist()
ok.show()

spark.sql("show databases").show(30, truncate=False)